# Part A — Dataset Exploration (Shopee Product Matching)

**Goal:** understand the data *before* building any model.
We answer five questions:
1. What is in the dataset and what does each column mean?
2. How big is it, and what do the product groups look like?
3. Are there duplicate images / titles?
4. What do "same product" and "different product" pairs look like — and when is it hard to tell them apart?
5. Which challenges will a machine-learning model face?

**How to read this notebook:** every code cell is followed by a short note on *what it shows*.
The bottom of the notebook has a section where you write your own observations after looking at the real outputs.

## 1. Setup — imports, settings and finding the dataset

This cell loads the libraries we need and finds the Shopee dataset automatically
(on Kaggle, Google Colab or your own computer — see `SETUP_GUIDE.md` for how to add the dataset).

In [ ]:
import os, re, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 100

SEED = 42                      # a fixed "random seed" so you get the same results every time you run
random.seed(SEED)
np.random.seed(SEED)

# ---------------- Settings you may want to change ----------------
N_GROUPS = 3000                # how many product groups to use (3000 groups ≈ 9,000 listings).
                               # Smaller = faster (use ~500 if you have no GPU). None = whole dataset (slow, needs lots of RAM).
RESULTS_DIR = "results"        # plots and tables are saved here
os.makedirs(RESULTS_DIR, exist_ok=True)


def find_data_dir():
    """Look for the folder that contains train.csv (works on Kaggle, Colab and a local computer)."""
    if os.environ.get("SHOPEE_DIR"):                       # you can force a path with this variable
        return os.environ["SHOPEE_DIR"]
    candidates = [
        "/kaggle/input/shopee-product-matching",           # Kaggle notebook (competition added via 'Add Input')
        "/kaggle/input/competitions/shopee-product-matching",
        "/content/shopee",                                 # Google Colab (see SETUP_GUIDE.md)
        "./shopee-product-matching", "./data", "../data", "../shopee-product-matching",
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "train.csv")):
            return c
    for root, dirs, files in os.walk("/kaggle/input"):     # last resort: search everything under /kaggle/input
        if "train.csv" in files:
            return root
    raise FileNotFoundError("train.csv not found. Follow the steps in SETUP_GUIDE.md to add the dataset.")


DATA_DIR = find_data_dir()
IMG_DIR = os.path.join(DATA_DIR, "train_images")
print("Using data from:", DATA_DIR)

## 2. Load the files and understand the columns

The Kaggle dataset contains:
* `train.csv` — ~34,000 product listings **with** the answer (`label_group`). We use only this file.
* `train_images/` — one photo per listing (file name = `image` column).
* `test.csv` / `test_images/` — only 3 dummy rows on Kaggle (the real test set is hidden), so we do not use them.

| Column | Meaning |
|---|---|
| `posting_id` | unique id of one listing (one seller's post) |
| `image` | file name of the listing's photo inside `train_images/` |
| `image_phash` | a 64-bit "perceptual hash" (a fingerprint of how the image *looks*, written as 16 hex characters). Similar-looking images have similar hashes |
| `title` | the text title written by the seller (noisy, mixed English / Indonesian / Malay) |
| `label_group` | **the answer**: listings with the same `label_group` are the same product |

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
print("train.csv shape (rows, columns):", df.shape)
display(df.head())
print("\nColumn types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isna().sum())

test_path = os.path.join(DATA_DIR, "test.csv")
if os.path.exists(test_path):
    print("\ntest.csv (note: no label_group column — the answer is hidden):")
    display(pd.read_csv(test_path))

**What a "product group" is and how matching is represented**

There is no table of pairs. Instead every listing carries a group id (`label_group`).
*Two listings match if and only if they have the same `label_group`.*
This is an **equivalence relation**: if A matches B and B matches C then A matches C (they all share one id).
So the whole matching problem can also be seen as **clustering** listings into groups.
(In the Kaggle submission file, each posting lists the `posting_id`s of all its matches, including itself.)

## 3. Basic statistics

In [ ]:
group_sizes = df.groupby("label_group").size()

n_listings = len(df)
n_groups = df["label_group"].nunique()
true_pairs = int((group_sizes * (group_sizes - 1) // 2).sum())          # number of matching pairs
all_pairs = n_listings * (n_listings - 1) // 2                          # number of possible pairs

stats = {
    "Number of listings": n_listings,
    "Number of product groups (unique label_group)": n_groups,
    "Unique image files": df["image"].nunique(),
    "Unique image hashes (image_phash)": df["image_phash"].nunique(),
    "Unique titles (exact text)": df["title"].nunique(),
    "Unique titles (lower-cased, trimmed)": df["title"].str.lower().str.strip().nunique(),
    "Smallest group size": int(group_sizes.min()),
    "Largest group size": int(group_sizes.max()),
    "Average group size": round(group_sizes.mean(), 2),
    "Median group size": float(group_sizes.median()),
    "Pairs of listings that really match": true_pairs,
    "All possible pairs of listings": all_pairs,
    "Share of pairs that match": f"{100 * true_pairs / all_pairs:.4f} %",
}
display(pd.Series(stats, name="value").to_frame())

**Why this matters:** only a tiny fraction of all possible pairs are real matches. So the problem is extremely
**imbalanced** — a model that always says "not a match" would be right >99.9 % of the time (and useless).
That is why we use precision / recall / F1 (Part B) instead of plain accuracy.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

vc = group_sizes.value_counts().sort_index()
vc_small = vc[vc.index <= 20]
ax[0].bar(vc_small.index, vc_small.values, color="tab:blue")
ax[0].set_yscale("log")
ax[0].set_xlabel("number of listings in the group (sizes above 20 not shown)")
ax[0].set_ylabel("number of groups (log scale)")
ax[0].set_title("Distribution of product-group sizes")

biggest = group_sizes.sort_values(ascending=False).head(10)
ax[1].barh([str(g) for g in biggest.index][::-1], biggest.values[::-1], color="tab:orange")
ax[1].set_xlabel("listings in the group")
ax[1].set_title("The 10 biggest groups (label_group ids)")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "A_group_sizes.png"), bbox_inches="tight")
plt.show()

print("Share of groups that have exactly 2 listings: %.1f %%" % (100 * (group_sizes == 2).mean()))
print("Share of groups with more than 10 listings:   %.1f %%" % (100 * (group_sizes > 10).mean()))
display(group_sizes.describe().to_frame("group size"))

**What it shows:** most groups are small (a handful of listings) but there is a long tail of very large groups.
Big groups produce many matching pairs, so they dominate the number of "positive" examples,
while small groups are numerous. A matching system must work for both: finding the 1 partner of a pair
*and* all 40 members of a big group.

## 4. Duplicates — images and titles

In [ ]:
# 1) The same image FILE used by several listings
g_per_image = df.groupby("image")["label_group"].nunique()
n_per_image = df["image"].value_counts()
print("Image files used by more than one listing:", int((n_per_image > 1).sum()))
print("   ...of which shared by listings of DIFFERENT products:", int((g_per_image > 1).sum()))

# 2) The same perceptual hash (visually identical or nearly identical images)
g_per_phash = df.groupby("image_phash")["label_group"].nunique()
n_per_phash = df["image_phash"].value_counts()
print("\nphash values used by more than one listing:", int((n_per_phash > 1).sum()))
print("   ...of which shared by listings of DIFFERENT products:", int((g_per_phash > 1).sum()))

# 3) The same title text
df["title_norm"] = df["title"].str.lower().str.strip()
g_per_title = df.groupby("title_norm")["label_group"].nunique()
n_per_title = df["title_norm"].value_counts()
print("\nTitles (lower-cased) used by more than one listing:", int((n_per_title > 1).sum()))
print("   ...of which shared by listings of DIFFERENT products:", int((g_per_title > 1).sum()))

fig, ax = plt.subplots(figsize=(8, 4))
names = ["listings", "unique image files", "unique phash", "unique titles"]
values = [len(df), df["image"].nunique(), df["image_phash"].nunique(), df["title_norm"].nunique()]
ax.bar(names, values, color=["gray", "tab:blue", "tab:green", "tab:orange"])
for k, v in enumerate(values):
    ax.text(k, v, f"{v:,}", ha="center", va="bottom")
ax.set_title("Listings vs unique images / hashes / titles")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "A_duplicates.png"), bbox_inches="tight")
plt.show()

**What it shows:** there are fewer unique `image_phash` values than unique image files, i.e. many *different files*
look (almost) identical. The most interesting numbers are the ones marked "DIFFERENT products": if identical images or
identical titles can belong to different products, then **an exact duplicate is not a guarantee of a match**.
(Compare these counts with the small ones from the "same group" case in the section 8 examples.)

## 5. Title length, language and noise

In [ ]:
df["title_chars"] = df["title"].str.len()
df["title_words"] = df["title"].str.split().str.len()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(df["title_chars"], bins=50, color="tab:blue")
ax[0].set_xlabel("title length (characters)"); ax[0].set_ylabel("number of listings")
ax[0].set_title("Title length in characters")
ax[1].hist(df["title_words"], bins=range(0, 40), color="tab:green")
ax[1].set_xlabel("title length (words)"); ax[1].set_title("Title length in words")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "A_title_length.png"), bbox_inches="tight")
plt.show()
display(df[["title_chars", "title_words"]].describe().round(1))

In [ ]:
from collections import Counter

word_counts = Counter(w for t in df["title"].str.lower() for w in re.findall(r"[a-zA-Z]+", t))
top_words = word_counts.most_common(30)

plt.figure(figsize=(9, 6))
plt.barh([w for w, c in top_words][::-1], [c for w, c in top_words][::-1], color="tab:purple")
plt.title("30 most common words in titles")
plt.xlabel("number of occurrences")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "A_top_words.png"), bbox_inches="tight")
plt.show()

In [ ]:
t = df["title"].fillna("")
promo_words = r"murah|promo|ready|grosir|original|termurah|diskon|sale|free|ongkir|bestseller|hot|new"
indo_words = r"\b(?:dan|untuk|dengan|yang|murah|baju|sepatu|tas|wanita|pria|anak|kaos|celana|isi|bahan)\b"
noise = pd.Series({
    "contains non-ASCII characters (emoji, accents, other scripts)": t.str.contains(r"[^\x00-\x7F]", na=False).mean(),
    "contains digits": t.str.contains(r"\d", na=False).mean(),
    "contains a size / quantity (10ml, 2pcs, 500gr, xl, 128gb ...)": t.str.lower().str.contains(r"\d+\s?(?:ml|gr|g|kg|pcs|cm|mm|gb|l)\b", na=False).mean(),
    "has a WORD IN CAPITALS (3+ letters)": t.str.contains(r"\b[A-Z]{3,}\b", na=False).mean(),
    "whole title in capitals": ((t == t.str.upper()) & (t.str.len() > 0)).mean(),
    "contains promo / marketing words": t.str.lower().str.contains(promo_words, na=False).mean(),
    "contains common Indonesian / Malay words": t.str.lower().str.contains(indo_words, na=False).mean(),
}) * 100
display(noise.round(1).to_frame("% of titles"))

**What it shows:** titles are *not* clean product names. They mix languages (English + Indonesian/Malay), contain marketing words
("murah", "promo", "original"), sizes and quantities, capital letters and symbols. Two sellers describing the *same* product will
rarely write the same words — and two *different* variants (e.g. 250 ml vs 500 ml) can differ by a single token.

## 6. Image properties (on a random sample of 1,000 images)

In [ ]:
sample_files = df["image"].sample(min(1000, len(df)), random_state=0)
sizes = []
for f in sample_files:
    try:
        sizes.append(Image.open(os.path.join(IMG_DIR, f)).size)     # only reads the header — fast
    except Exception:
        pass
img_sizes = pd.DataFrame(sizes, columns=["width", "height"])
img_sizes["aspect (w/h)"] = img_sizes["width"] / img_sizes["height"]

fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].hist(img_sizes["width"], bins=30); ax[0].set_title("image width (px)")
ax[1].hist(img_sizes["height"], bins=30); ax[1].set_title("image height (px)")
ax[2].hist(img_sizes["aspect (w/h)"], bins=30); ax[2].set_title("aspect ratio (width / height)")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "A_image_sizes.png"), bbox_inches="tight")
plt.show()
print("Share of images that are NOT square: %.1f %%" % (100 * (img_sizes["width"] != img_sizes["height"]).mean()))
display(img_sizes.describe().round(1))

**What it shows:** images come in different sizes and shapes. A neural network needs a fixed input size (e.g. 224 × 224),
so we must resize — and squashing or cropping non-square images can distort or cut off the product (Part C).

## 7. What makes two listings the same product? Titles and images, pair by pair

To compare titles and images fairly we look at **all pairs** of listings in a random sample of 2,000 product groups
(all members of each group are included). For every pair we measure
* **title similarity** = TF-IDF cosine similarity (0 = no words in common, 1 = identical words),
* **image distance** = Hamming distance between the two `image_phash` values (0 = identical hash, ~32 = unrelated images).

Then we compare the two classes: *pairs that match* vs *pairs that do not*.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

rng = np.random.RandomState(0)
chosen_groups = rng.choice(df["label_group"].unique(), size=min(2000, df["label_group"].nunique()), replace=False)
sub_a = df[df["label_group"].isin(chosen_groups)].reset_index(drop=True)
labels_a = sub_a["label_group"].values
n = len(sub_a)
print(f"Sample: {n:,} listings, {sub_a['label_group'].nunique():,} groups, {n * (n - 1) // 2:,} pairs")

same = labels_a[:, None] == labels_a[None, :]            # same[i, j] = True if i and j are the same product
upper = np.triu(np.ones((n, n), dtype=bool), k=1)         # look at every pair only once (i < j)
is_match = same[upper]

# --- title similarity (TF-IDF cosine) ---
X_a = TfidfVectorizer().fit_transform(sub_a["title"].fillna(""))
S_text = (X_a @ X_a.T).toarray().astype(np.float32)
title_sim = S_text[upper]


# --- image distance (Hamming distance between the 64-bit phash values) ---
def phash_to_bits(hashes):
    """'e9b9d5b0a4a4b6b5' -> array of 64 zeros/ones"""
    return np.array([[int(b) for b in format(int(h, 16), "064b")] for h in hashes], dtype=np.float32)


bits_a = phash_to_bits(sub_a["image_phash"])
ones = bits_a.sum(axis=1)
ham = ones[:, None] + ones[None, :] - 2 * (bits_a @ bits_a.T)    # number of differing bits (exact for 0/1 vectors)
ham_vals = ham[upper]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(title_sim[~is_match], bins=40, range=(0, 1), density=True, histtype="step", lw=2, label="different products")
ax[0].hist(title_sim[is_match], bins=40, range=(0, 1), density=True, histtype="step", lw=2, label="same product")
ax[0].set_yscale("log"); ax[0].set_xlabel("title similarity (TF-IDF cosine)"); ax[0].set_ylabel("density (log)")
ax[0].set_title("Titles: matching vs non-matching pairs"); ax[0].legend()

ax[1].hist(ham_vals[~is_match], bins=33, range=(0, 64), density=True, histtype="step", lw=2, label="different products")
ax[1].hist(ham_vals[is_match], bins=33, range=(0, 64), density=True, histtype="step", lw=2, label="same product")
ax[1].set_xlabel("phash Hamming distance (lower = more similar image)"); ax[1].set_ylabel("density")
ax[1].set_title("Images: matching vs non-matching pairs"); ax[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "A_pair_distributions.png"), bbox_inches="tight")
plt.show()

In [ ]:
n_pos = int(is_match.sum())
rows = []
for k in [0, 2, 4, 6, 8, 10, 12]:
    called_match = ham_vals <= k
    tp = int((called_match & is_match).sum())
    rows.append({"rule": f"image: phash distance <= {k}", "pairs called 'match'": int(called_match.sum()),
                 "correct": tp, "precision": round(tp / max(called_match.sum(), 1), 3), "recall": round(tp / n_pos, 3)})
for t_ in [0.3, 0.5, 0.7, 0.9]:
    called_match = title_sim >= t_
    tp = int((called_match & is_match).sum())
    rows.append({"rule": f"title: TF-IDF cosine >= {t_}", "pairs called 'match'": int(called_match.sum()),
                 "correct": tp, "precision": round(tp / max(called_match.sum(), 1), 3), "recall": round(tp / n_pos, 3)})
display(pd.DataFrame(rows))
print(f"Total true matching pairs in this sample: {n_pos:,}")
print("True pairs whose titles share NO word at all (similarity = 0): %.1f %%" % (100 * (title_sim[is_match] == 0).mean()))
print("True pairs whose images are far apart (phash distance > 16):   %.1f %%" % (100 * (ham_vals[is_match] > 16).mean()))

**How to read this:**
* If the "same product" curve and the "different products" curve **overlap**, a single feature (title *or* image) cannot separate them perfectly.
* The table is a mini-matching system with a single rule. Look at the trade-off: a strict rule (small distance / high similarity)
  has high precision but low recall; a loose rule finds more true matches but also many wrong ones.
* The last two lines tell us how many true matches are *invisible* to titles alone (no shared word) or to images alone (very different picture).
  These numbers answer: *"Can titles alone / images alone reliably determine a match?"* — and justify combining both in the Finale.

## 8. Look at real examples

Helper functions to draw listings (image + title).

In [ ]:
import textwrap


def load_img(fname):
    try:
        return Image.open(os.path.join(IMG_DIR, fname)).convert("RGB")
    except Exception:
        return Image.new("RGB", (200, 200), (200, 200, 200))     # grey placeholder if the file is missing


def short_title(s, width=30, max_chars=80):
    return textwrap.fill(textwrap.shorten(str(s), max_chars, placeholder="…"), width)


def show_listings(rows, ncols=4, heading=None):
    """Draw several listings (rows of df) in a grid: image on top, title as caption."""
    rows = rows.reset_index(drop=True)
    nrows = int(np.ceil(len(rows) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.2, nrows * 3.5))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, (_, r) in zip(axes, rows.iterrows()):
        ax.imshow(load_img(r["image"]))
        ax.set_title(short_title(r["title"]), fontsize=8)
    if heading:
        fig.suptitle(heading, fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()


def show_pairs(pairs, data, pairs_per_row=2):
    """pairs = list of (row_i, row_j, note). Draws the two listings of each pair side by side."""
    nrows = int(np.ceil(len(pairs) / pairs_per_row))
    ncols = 2 * pairs_per_row
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.1, nrows * 3.6))
    axes = np.array(axes).reshape(nrows, ncols)
    for ax in axes.ravel():
        ax.axis("off")
    for p, (i, j, note) in enumerate(pairs):
        r, c = divmod(p, pairs_per_row)
        same_group = data.iloc[i]["label_group"] == data.iloc[j]["label_group"]
        for k, idx in enumerate((i, j)):
            ax = axes[r, 2 * c + k]
            ax.imshow(load_img(data.iloc[idx]["image"]))
            ax.set_title(short_title(data.iloc[idx]["title"], 28), fontsize=7.5)
        axes[r, 2 * c].text(1.0, -0.06, f"{note}\nsame product: {'YES' if same_group else 'NO'}", transform=axes[r, 2 * c].transAxes,
                            ha="center", va="top", fontsize=8, color="darkgreen" if same_group else "darkred")
    plt.tight_layout()
    plt.show()

### 8.1 Several listings of the SAME product

In [ ]:
rng = np.random.RandomState(1)
mid_groups = group_sizes[(group_sizes >= 4) & (group_sizes <= 8)].index.values
for g in rng.choice(mid_groups, size=3, replace=False):
    show_listings(df[df["label_group"] == g], heading=f"label_group {g}: {int(group_sizes[g])} listings of the same product")

**Why this is hard:** look at how differently the same product is described (word order, extra promo words, different
languages/abbreviations) and photographed (different angle, background, crop, sometimes a different picture entirely).
Identity is defined by the *product*, not by the words or the pixels.

### 8.2 Same product, very DIFFERENT images
We rank groups (3+ listings) by the average phash distance between their images — the bigger, the more different the pictures.

In [ ]:
bits_all = phash_to_bits(df["image_phash"])
spread = {}
for g, idx in df.groupby("label_group").indices.items():
    if len(idx) >= 3:
        b = bits_all[idx]
        o = b.sum(axis=1)
        d = o[:, None] + o[None, :] - 2 * (b @ b.T)
        spread[g] = d[np.triu_indices(len(idx), k=1)].mean()
spread = pd.Series(spread).sort_values(ascending=False)
print("Average phash distance inside a group — median over groups: %.1f | most different group: %.1f (random unrelated images ≈ 32)"
      % (spread.median(), spread.iloc[0]))
for g in spread.index[:2]:
    show_listings(df[df["label_group"] == g].head(8), heading=f"label_group {g}: same product, images differ a lot (avg distance {spread[g]:.1f})")

**Why this is hard:** an image model that only looks at overall appearance will say "different product" here.
Sellers photograph the product alone, in a package, worn by a model, as a collage, with text overlaid, etc.

### 8.3 SIMILAR titles but DIFFERENT products
From all pairs of different products in our 2,000-group sample, we take the pairs with the highest title similarity.

In [ ]:
S_diff = np.triu(S_text, k=1)           # each pair once
S_diff[same] = -1                       # remove true matches -> only DIFFERENT-product pairs remain
flat = S_diff.ravel()
k = min(400, flat.size - 1)
top = np.argpartition(flat, -k)[-k:]
top = top[np.argsort(-flat[top])]       # sorted: most similar first
ii, jj = np.unravel_index(top, S_diff.shape)
sims = flat[top]

pairs = [(ii[p], jj[p], f"title similarity = {sims[p]:.2f}") for p in range(4)]                  # the very top
mid = np.where((sims >= 0.6) & (sims < 0.95))[0]                                                   # a few "not identical" ones
if len(mid):
    for p in np.random.RandomState(2).choice(mid, size=min(4, len(mid)), replace=False):
        pairs.append((ii[p], jj[p], f"title similarity = {sims[p]:.2f}"))
show_pairs(pairs, sub_a)
print("Number of DIFFERENT-product pairs with title similarity >= 0.8:", int((flat >= 0.8).sum()))

**Why this is hard:** a title can be (nearly) identical for products that differ in *size, colour, quantity, model number or flavour*.
One changed token ("128GB" vs "64GB", "250ml" vs "500ml") changes the product but barely changes a similarity score.
Also note that exact duplicate titles may belong to different groups (generic titles).

### 8.4 VISUALLY similar images but DIFFERENT products

In [ ]:
close = upper & ~same & (ham <= 6)
ci, cj = np.where(close)
print("Different-product pairs whose images are nearly identical (phash distance <= 6):", len(ci))
if len(ci) > 0:
    order = np.argsort(ham[ci, cj])
    pairs = [(ci[o], cj[o], f"phash distance = {int(ham[ci[o], cj[o]])}") for o in order[:4]]
    rest = order[4:]
    if len(rest):
        for o in np.random.RandomState(3).choice(rest, size=min(4, len(rest)), replace=False):
            pairs.append((ci[o], cj[o], f"phash distance = {int(ham[ci[o], cj[o]])}"))
    show_pairs(pairs, sub_a)
else:
    print("None found in this sample — try a bigger sample or a larger distance (e.g. <= 10).")

**Why this is hard:** products of the same *family* (same shape, same packaging, different flavour/colour/size) look alike,
and sellers often reuse the same stock photo for different variants. Images alone cannot tell them apart —
the **text** ("strawberry" vs "vanilla") often carries the deciding detail.

### 8.5 Noisy or incomplete listings

In [ ]:
t = df["title"].fillna("")
df["alnum_ratio"] = t.apply(lambda s: sum(c.isalnum() or c.isspace() for c in s) / max(len(s), 1))

print("Shortest titles:")
display(df.nsmallest(8, "title_chars")[["title", "title_chars", "label_group"]])
print("Titles with the most symbols / emoji (lowest share of letters+digits):")
display(df.nsmallest(8, "alnum_ratio")[["title", "alnum_ratio", "label_group"]])
print("Longest titles (keyword stuffing):")
display(df.nlargest(5, "title_chars")[["title", "title_chars"]])

shortest = df.nsmallest(1, "title_chars").iloc[0]
show_listings(df[df["label_group"] == shortest["label_group"]].head(8),
              heading=f"The group of the shortest title ('{shortest['title']}') — how would you find its partners from the title alone?")

**Why this is hard:** very short titles carry almost no information; titles full of symbols/emoji or stuffed with keywords
are dominated by noise that is unrelated to the product identity. For such listings the image (or other signals) must carry the decision.

## 9. Challenges a machine-learning model will face

| # | Challenge | Evidence in this notebook | Why it hurts | Possible remedy |
|---|---|---|---|---|
| 1 | **Noisy titles** (promo words, symbols, capitals, keyword stuffing) | Sections 5, 8.5 | Many words say nothing about the product | text cleaning, TF-IDF weighting, character n-grams |
| 2 | **Mixed languages** (English + Indonesian/Malay) and abbreviations | Section 5 (top words, language table) | Same meaning, different words | multilingual embeddings, char n-grams |
| 3 | **Variants of the same family** (size / colour / quantity) | Section 8.3 | Very similar text *and* image but different product | pay attention to numbers & attribute words; combine modalities |
| 4 | **Same product, different pictures** | Section 8.2, phash histogram | Pixel/appearance similarity fails | semantic image features (CLIP), use text as backup |
| 5 | **Visually similar, different products** | Section 8.4, phash precision table | Images alone give false matches | combine with text |
| 6 | **Very short / missing information** | Section 8.5 | Nothing to compare | rely on the other modality |
| 7 | **Extreme class imbalance** | Section 3 (share of matching pairs) | Accuracy is meaningless; need threshold tuning and F1 | retrieval + threshold, hard-negative mining |
| 8 | **Huge variation in group size** (2 … ~50) | Section 3 | A single global threshold fits small and large groups differently | per-group/adaptive thresholds, clustering |
| 9 | **Exact duplicates are not always matches** | Section 4 | "Identical" can still be a different product | do not trust any single signal blindly |
| 10 | **Scale** (34k × 34k comparisons) | Section 3 | Memory / time | embeddings + nearest-neighbour search (FAISS) |

(Check each row against what *you* saw in the real plots and edit this table if your data tells a different story.)

## 10. Answers to the guiding questions

* **What makes two listings belong to the same product?** They carry the same `label_group`: they describe the same underlying product
  regardless of seller, title wording or photo.
* **Can product titles alone reliably determine whether two listings match?** Not perfectly — see the title histogram and the last lines
  of section 7: some true pairs share no words, while some different products have almost identical titles.
* **Can images alone reliably determine it?** Not perfectly either — some true pairs have very different pictures, and many different products
  have near-identical pictures (section 7 table, section 8.4).
* **Which examples are hard?** Variants (size/colour), generic or very short titles, same product with different photos,
  listings with stock photos.
* **What information is useful?** Title text (words, numbers), image content (semantic embedding), image hash (cheap exact-duplicate cue),
  and structural cues such as numbers/units in the title. Titles and images fail in *different* cases, so a combination should work better than either alone.

## 11. Key numbers for your README (copy these after running)

In [ ]:
summary = {
    "listings": n_listings,
    "product groups": n_groups,
    "average group size": round(group_sizes.mean(), 2),
    "largest group": int(group_sizes.max()),
    "share of pairs that match (%)": round(100 * true_pairs / all_pairs, 4),
    "unique image files": df["image"].nunique(),
    "unique phash": df["image_phash"].nunique(),
    "phash values shared by DIFFERENT products": int((g_per_phash > 1).sum()),
    "titles shared by DIFFERENT products": int((g_per_title > 1).sum()),
    "median title length (chars)": float(df["title_chars"].median()),
    "true pairs with zero shared title words (%, sample)": round(100 * (title_sim[is_match] == 0).mean(), 1),
    "true pairs with phash distance > 16 (%, sample)": round(100 * (ham_vals[is_match] > 16).mean(), 1),
}
summary_df = pd.Series(summary, name="value").to_frame()
display(summary_df)
summary_df.to_csv(os.path.join(RESULTS_DIR, "A_key_numbers.csv"))

## 12. My observations and conclusions  ✍️ (write these yourself after running)

*Replace each line below with what you actually saw — your own words are what the interviewer will ask about.*

1. **Dataset size & structure:** …
2. **Group sizes:** …
3. **Duplicates:** …
4. **Titles:** (length, languages, noise) …
5. **Images:** (sizes, variety) …
6. **Titles alone:** can / cannot decide because … (cite one number from section 7)
7. **Images alone:** can / cannot decide because … (cite one number from section 7)
8. **Hardest examples I found:** …
9. **My plan for Parts B, C and the Finale:** (e.g. text with TF-IDF + embeddings, image with CLIP, then fuse the two) …